# 문서 분석 · 번역 · TTS — Linux/Jupyter 수정본

이 노트북은 **실제 NHN Cloud T4 실행 결과(Python 3.13.12, CUDA 드라이버 13.0)**를 반영해 수정했습니다. 현재 폴더에 `app.py`와 `tts_api_server.py`가 있어야 합니다. 모든 셀을 위에서 아래 순서로 실행하세요.

## 수정한 오류

- Chatterbox는 공식적으로 Python 3.11에서 개발·검증되었습니다. Python 3.13 환경에서 `PerthImplicitWatermarker`가 `None`이 되는 오류가 발생했으므로, Miniforge의 Python 3.11 환경을 별도로 만듭니다.
- TTS 모델 다운로드가 끝나기 전에 상태 확인을 하던 문제를, 최대 10분까지 기다리고 실패 시 로그를 출력하는 방식으로 바꿨습니다.
- Gradio 실행 셀에 없던 `os` 가져오기를 포함해, 독립 실행 가능한 셀로 만들었습니다.

In [5]:
# 실제 클라우드 환경 확인. T4와 여유 공간이 보이는지 확인합니다.
!nvidia-smi
!df -h .
!$HOME/miniforge3/bin/conda --version

Thu Aug 27 16:00:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:00:06.0 Off |                    0 |
| N/A   36C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# Python 3.11 환경을 각각 한 번만 만듭니다. 이미 있으면 conda가 재사용합니다.
from pathlib import Path
import subprocess

CONDA = Path.home() / 'miniforge3' / 'bin' / 'conda'
if not CONDA.exists():
    raise RuntimeError('Miniforge를 찾지 못했습니다. Jupyter 서버의 Python 3.11 또는 Miniforge 설치가 필요합니다.')

def run(*args):
    print('+', ' '.join(map(str, args)))
    subprocess.run(list(map(str, args)), check=True)

run(CONDA, 'create', '-y', '-n', 'document-app', 'python=3.11', 'pip')
run(CONDA, 'create', '-y', '-n', 'document-tts', 'python=3.11', 'pip')

+ /home/ubuntu/miniforge3/bin/conda create -y -n document-app python=3.11 pip
Retrieving notices: done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/ubuntu/miniforge3/envs/document-app

  added / updated specs:
    - pip
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |      hda65f42_10         252 KB  conda-forge
    ca-certificates-2026.7.22  |       hbd8a1cb_0         129 KB  conda-forge
    icu-78.3                   |  py310h44b86e0_2        13.8 MB  conda-forge
    ld_impl_linux-64-2.46.1    |default_hbd61a6d_102         728 KB  conda-forge
    libexpat-2.8.1             |       hecca717_1          76 KB  conda-forge
    libffi-3.7.0               |       h81df57d_1          66 KB  con



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.7.1

Please update conda by running

    $ conda update -n base -c conda-forge conda




python-3.11.16       | 29.5 MB   | 5          |   5% 
icu-78.3             | 13.8 MB   | #          |  10% 

libstdcxx-16.2.0     | 6.3 MB    | ##5        |  25% 


tk-8.6.13            | 3.4 MB    | #######7   |  77% 



openssl-3.6.4        | 3.1 MB    | #####8     |  58% 


tk-8.6.13            | 3.4 MB    | ########## | 100% 



python-3.11.16       | 29.5 MB   | ##         |  20% 
icu-78.3             | 13.8 MB   | ####2      |  42% 




pip-26.2.1           | 1.1 MB    | 1          |   1% 





libgcc-16.2.0        | 1.0 MB    | 1          |   2% 




pip-26.2.1           | 1.1 MB    | ########## | 100% 





libgcc-16.2.0        | 1.0 MB    | ########## | 100% 






libsqlite-3.53.4     | 952 KB    | 1          |   2% 







python-3.11.16       | 29.5 MB   | ###1       |  31% 
icu-78.3             | 13.8 MB   | ######5    |  66% 







ncurses-6.6          | 890 KB    | ########## | 100% 






libsqlite-3.53.4     | 952 KB    | ########## | 100% 








ld_impl_linux-64-2.



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.7.1

Please update conda by running

    $ conda update -n base -c conda-forge conda




done
Verifying transaction: done
Executing transaction: done
#
# To activate this environment, use
#
#     $ conda activate document-tts
#
# To deactivate an active environment, use
#
#     $ conda deactivate



In [7]:
# 의존성은 실제 T4/드라이버 580 환경과 호환되는 CUDA 12.4 PyTorch로 고정합니다.
# 최초 1회에만 실행하며 다운로드 용량이 큽니다.
run(CONDA, 'run', '-n', 'document-app', 'python', '-m', 'pip', 'install', '--upgrade', 'pip')
run(CONDA, 'run', '-n', 'document-app', 'pip', 'install', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124')
run(CONDA, 'run', '-n', 'document-app', 'pip', 'install', 'accelerate==1.3.0', 'gradio==5.17.1', 'python-docx==1.1.2', 'requests==2.32.3', 'sentencepiece==0.2.0', 'transformers==4.49.0')

run(CONDA, 'run', '-n', 'document-tts', 'python', '-m', 'pip', 'install', '--upgrade', 'pip', 'wheel', 'setuptools==80.9.0')
run(CONDA, 'run', '-n', 'document-tts', 'pip', 'install', 'numpy==1.26.4')
run(CONDA, 'run', '-n', 'document-tts', 'pip', 'install', 'torch==2.6.0', 'torchaudio==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124')
run(CONDA, 'run', '-n', 'document-tts', 'pip', 'install', 'chatterbox-tts==0.1.7', 'fastapi==0.115.12', 'uvicorn[standard]==0.34.0')

+ /home/ubuntu/miniforge3/bin/conda run -n document-app python -m pip install --upgrade pip
+ /home/ubuntu/miniforge3/bin/conda run -n document-app pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/24.6 MB ? eta -:--:--
     ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/24.6 MB 27.0 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 24.1/24.6 MB 80.5 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/883.7 kB ? eta -:--:--

In [ ]:
# Perth 복구: setuptools 80.9.0. 현재 환경에 이미 설치했다면 이 셀만 먼저 실행하세요.
run(CONDA, 'run', '-n', 'document-tts', 'pip', 'install', '--force-reinstall', 'setuptools==80.9.0')
run(CONDA, 'run', '-n', 'document-tts', 'python', '-c', "import pkg_resources, perth; assert callable(perth.PerthImplicitWatermarker); print('Perth OK:', perth.PerthImplicitWatermarker)")

In [10]:
run(CONDA, 'run', '-n', 'document-app', 'python', '-c', "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))")

run(CONDA, 'run', '-n', 'document-tts', 'python', '-c', "import torch; import perth; assert callable(perth.PerthImplicitWatermarker); assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0), 'Perth OK')")

+ /home/ubuntu/miniforge3/bin/conda run -n document-app python -c import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0))
2.6.0+cu124 Tesla T4
+ /home/ubuntu/miniforge3/bin/conda run -n document-tts python -c import torch; import perth; assert callable(perth.PerthImplicitWatermarker); assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0), 'Perth OK')


Traceback (most recent call last):
  File "<string>", line 1, in <module>
AssertionError
ERROR conda.cli.main_run:execute(142): `conda run python -c import torch; import perth; assert callable(perth.PerthImplicitWatermarker); assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0), 'Perth OK')` failed. (See above for error)


CalledProcessError: Command '['/home/ubuntu/miniforge3/bin/conda', 'run', '-n', 'document-tts', 'python', '-c', "import torch; import perth; assert callable(perth.PerthImplicitWatermarker); assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name(0), 'Perth OK')"]' returned non-zero exit status 1.

In [11]:
run(CONDA, 'run', '-n', 'document-tts', 'python', '-c', "import torch; print('CUDA:', torch.cuda.is_available()); print('Torch:', torch.__version__)")

+ /home/ubuntu/miniforge3/bin/conda run -n document-tts python -c import torch; print('CUDA:', torch.cuda.is_available()); print('Torch:', torch.__version__)
CUDA: True
Torch: 2.6.0+cu124


In [12]:
run(CONDA, 'run', '-n', 'document-tts', 'python', '-c', "import perth; print(perth.PerthImplicitWatermarker); print(callable(perth.PerthImplicitWatermarker))")

+ /home/ubuntu/miniforge3/bin/conda run -n document-tts python -c import perth; print(perth.PerthImplicitWatermarker); print(callable(perth.PerthImplicitWatermarker))
None
False


In [ ]:
# TTS 서버를 시작한 뒤, 모델 다운로드와 준비가 끝날 때까지 기다립니다.
import os, time, requests
from pathlib import Path

if not Path('tts_api_server.py').exists():
    raise FileNotFoundError('현재 폴더에 tts_api_server.py가 없습니다. 프로젝트 파일을 함께 업로드하세요.')

tts_log = open('tts_server.log', 'w', buffering=1)
TTS_PROCESS = subprocess.Popen([str(CONDA), 'run', '--no-capture-output', '-n', 'document-tts', 'python', '-u', '-m', 'uvicorn', 'tts_api_server:app', '--host', '127.0.0.1', '--port', '8001'], stdout=tts_log, stderr=subprocess.STDOUT)

deadline = time.time() + 600
while time.time() < deadline:
    if TTS_PROCESS.poll() is not None:
        raise RuntimeError('TTS 서버가 종료되었습니다.\n' + Path('tts_server.log').read_text(errors='replace')[-4000:])
    try:
        status = requests.get('http://127.0.0.1:8001/health', timeout=5).json()
        if status.get('model_loaded'):
            print('TTS 준비 완료:', status); break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError('10분 안에 TTS가 준비되지 않았습니다.\n' + Path('tts_server.log').read_text(errors='replace')[-4000:])

In [ ]:
# Gradio 앱을 시작합니다. app.py의 TTS 호출은 내부 전용 TTS 주소로 연결됩니다.
if not Path('app.py').exists():
    raise FileNotFoundError('현재 폴더에 app.py가 없습니다. 프로젝트 파일을 함께 업로드하세요.')

app_log = open('gradio_app.log', 'w', buffering=1)
app_env = os.environ.copy()
app_env['TTS_API_URL'] = 'http://127.0.0.1:8001'
APP_PROCESS = subprocess.Popen([str(CONDA), 'run', '--no-capture-output', '-n', 'document-app', 'python', '-u', 'app.py'], stdout=app_log, stderr=subprocess.STDOUT, env=app_env)

deadline = time.time() + 90
while time.time() < deadline:
    if APP_PROCESS.poll() is not None:
        raise RuntimeError('Gradio 앱이 종료되었습니다.\n' + Path('gradio_app.log').read_text(errors='replace')[-4000:])
    try:
        if requests.get('http://127.0.0.1:7860/', timeout=5).ok:
            print('Gradio 준비 완료: http://<NHN-공인-IP>:7860'); break
    except requests.RequestException:
        pass
    time.sleep(3)
else:
    raise TimeoutError('Gradio 시작 시간이 초과되었습니다.\n' + Path('gradio_app.log').read_text(errors='replace')[-4000:])

## 접속

NHN 보안 그룹에서 **내 IP → TCP 7860**을 허용한 후, 로컬 브라우저에서 `http://<NHN-공인-IP>:7860`으로 접속합니다. TTS 포트 8001은 외부에 열지 않습니다.

문서 업로드 후 분석 요청을 입력하고 **분석 시작**을 누르세요. 분석 결과와 한국어·영어·일본어 번역, 각 언어의 WAV 음성이 표시됩니다.

In [ ]:
# 문제가 생겼을 때 마지막 100줄을 확인합니다.
!tail -n 100 tts_server.log
!tail -n 100 gradio_app.log

In [ ]:
# 작업 종료 시에만 실행합니다.
# APP_PROCESS.terminate()
# TTS_PROCESS.terminate()